**Подход — двухэтапный каскад**
1. **Кандидатогенерация**: объединённый пул из трёх источников: лексический BM25, плотный поиск, пул локация-категория по популярности;
2. **Реранкинг** пула градиентным бустингом CatBoost (YetiRank) до топ-50

**Результат:** локальная валидация Recall@50 = 0.7925, на бенчмарке = 0.7346

In [ ]:
%pip install pymorphy3 bm25s sentence-transformers

/usr/lib/python3.12/pty.py:95: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


In [ ]:
import gc, re, time
from functools import lru_cache
from pathlib import Path
from tqdm import tqdm

import numpy as np
import pandas as pd
import bm25s
from bm25s import BM25
import pymorphy3
from catboost import CatBoostRanker, Pool

DATA_DIR = Path("/kaggle/input/datasets/nicctmp/dataset-avito-bc")
ANSWER_PATH = Path("/kaggle/working/answer.csv")
MODEL_PATH = Path("/kaggle/working/catboost_reranker.cbm")

RANDOM_STATE = 42
TASK_TYPE = "GPU"

N_DENSE = 300 
N_BM25 = 500 
M_LOCCAT = 200
K_MAX = 500
CB_ITERATIONS = 800
CB_LEARNING_RATE = 0.12

QUERY_KEY_COLS = ["search_query", "search_location_id", "search_is_delivery_search",
                  "search_infm_params_text", "search_category"]

NUM_FEATURES = [
    "bm25_score", "bm25_rank", "in_bm25", "in_loccat",
    "dense_score", "dense_rank",
    "loc_match", "cat_match", "is_delivery", "q_len",
    "q_in_title_frac", "title_len",
    "item_rating", "log_reviews", "price_neg", "log_price",
    "rating_gap",
]
ALL_FEATURES = NUM_FEATURES + ["microcat"]
CAT_FEATURES = ["microcat"]


Лемматизация (pymorphy3) приводит слова к нормальной форме, чтобы лексически разные слова совпадали в индексе

In [3]:
morph = pymorphy3.MorphAnalyzer()
_clean = re.compile(r"[^а-яёa-z0-9]+")

@lru_cache(maxsize=2_000_000)
def lemma(w):
    return morph.parse(w)[0].normal_form

def tok(text):
    if not text:
        return []
    text = str(text).lower().replace("ё", "е")
    return [lemma(w) for w in _clean.split(text) if len(w) > 1]


## 2. Этап 1 - кандидатогенерация


In [ ]:
train_cols = ["search_query", "search_location_id", "search_is_delivery_search",
              "search_infm_params_text", "search_category", "item_id"]
train = pd.read_parquet(DATA_DIR / "train.parquet", columns=train_cols)
queries = pd.read_parquet(DATA_DIR / "benchmark_queries.parquet")
item_cols = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text",
             "item_location_id", "item_category_id", "item_microcat_id",
             "item_price", "item_rating", "item_rating_reviews_count"]
items = pd.read_parquet(DATA_DIR / "benchmark_items.parquet", columns=item_cols)

print(f"train: {len(train):,} пар | корпус: {len(items):,} | бенчмарк: {len(queries):,} запросов")


train: 497,673 пар | корпус: 189,212 | бенчмарк: 2,452 запросов


**Лексический поиск (BM25).** Документ = лемматизированный текст объявления с весами полей:
заголовок * 4, описание * 1, параметры * 2 (заголовок самый информативный, параметры точнее
длинного описания). Запрос = `search_query` + `search_infm_params_text`

In [ ]:
docs = [
    tok(t) * 4 + tok(d) + tok(p) * 2
    for t, d, p in tqdm(zip(items["item_title_raw"], items["item_description_raw"],
                            items["item_infm_params_text"]),
                        total=len(items), desc="лемматизация корпуса")
]
retriever = BM25()
retriever.index(docs)


лемматизация корпуса: 100%|██████████| 189212/189212 [02:24<00:00, 1306.73it/s]


BM25S Create Vocab:   0%|          | 0/189212 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/189212 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/189212 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/189212 [00:00<?, ?it/s]

индекс готов за 3.3 мин


**Плотный поиск (E5).** BM25 не ловит морфологические варианты и синонимы поэтому часть
подходящих объявлений вообще отсутствует в его топ-500 multilingual-e5-base кодирует заголовок + начало описания и запрос в векторы, косинусное расстояние для близости

In [6]:
from sentence_transformers import SentenceTransformer
import torch

EMB_MODEL = "intfloat/multilingual-e5-base"
emb = SentenceTransformer(EMB_MODEL, device="cuda:1",
                          model_kwargs={"torch_dtype": "float16"})

doc_texts = ["passage: " + str(t) + ". " + str(d)[:150]
             for t, d in zip(items["item_title_raw"], items["item_description_raw"])]
t0 = time.time()
doc_emb = emb.encode(doc_texts, batch_size=256, show_progress_bar=True,
                     normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
print(f"корпус E5: {doc_emb.shape}, {(time.time() - t0) / 60:.1f} мин")

def dense_retrieve(query_texts, k, doc_matrix):
    """top-k по косинусу для списка сырых текстов запросов (чанки — чтобы не раздуть память)."""
    q_emb = emb.encode(["query: " + q for q in query_texts], batch_size=512,
                       normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    tops, sims_top = [], []
    CH = 1024
    for s in range(0, len(q_emb), CH):
        sim = q_emb[s:s + CH] @ doc_matrix.T
        top = np.argpartition(-sim, k, axis=1)[:, :k]
        order = np.argsort(-np.take_along_axis(sim, top, axis=1), axis=1)
        tops.append(np.take_along_axis(top, order, axis=1))
        sims_top.append(np.take_along_axis(np.take_along_axis(sim, top, axis=1), order, axis=1))
    return np.vstack(tops), np.vstack(sims_top)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/740 [00:00<?, ?it/s]

корпус E5: (189212, 768), 4.6 мин


**Третий источник - локация-категория** Пользователь почти всегда выбирает
исполнителя в своей локации, поэтому объявления из связки
локация-категория, отсортированные по числу отзывов, тоже попадают в пул


In [ ]:
corpus_ids = items["item_id"].to_numpy()
id2row = {iid: i for i, iid in enumerate(corpus_ids)}

price = items["item_price"].map(float)
cf = {
    "ids": corpus_ids,
    "loc": items["item_location_id"].to_numpy(),
    "cat": items["item_category_id"].to_numpy(),
    "rating": items["item_rating"].fillna(-1).to_numpy(dtype=np.float32),
    "log_reviews": np.log1p(items["item_rating_reviews_count"].fillna(0)).to_numpy(dtype=np.float32),
    "price_neg": (price < 0).to_numpy(dtype=np.float32),
    "log_price": np.log1p(price.clip(lower=0)).to_numpy(dtype=np.float32),
    "title_sets": [frozenset(tok(t)) for t in items["item_title_raw"]],
}

cf["microcat_codes"] = np.unique(items["item_microcat_id"].astype(str), return_inverse=True)[1]


items_sorted = items.assign(_rev=items["item_rating_reviews_count"].fillna(0)
                            ).sort_values("_rev", ascending=False)
cf["loccat"] = (items_sorted.groupby(["item_location_id", "item_category_id"])["item_id"]
                .apply(list).to_dict())


In [ ]:
# пары, чей позитив есть в корпусе
valid = train[train["item_id"].isin(set(corpus_ids))].reset_index(drop=True)
print(f"валидационных пар: {len(valid):,}")

q_texts = [q + " " + (p or "") for q, p in zip(valid["search_query"], valid["search_infm_params_text"])]
q_tokens = [tok(t) or ["#"] for t in tqdm(q_texts, desc="лемматизация запросов")]
q_token_sets = [set(t) for t in q_tokens]

res_valid, scores_valid = retriever.retrieve(q_tokens, k=min(K_MAX, len(items)))


валидационных пар: 33,010


лемматизация запросов: 100%|██████████| 33010/33010 [00:01<00:00, 18251.23it/s] 


BM25S Retrieve:   0%|          | 0/33010 [00:00<?, ?it/s]

In [9]:
res_dense_v, sim_dense_v = dense_retrieve(q_texts, N_DENSE, doc_emb)

q_texts_b = [q + " " + (p or "") for q, p in
             zip(queries["search_query"], queries["search_infm_params_text"])]
res_dense_b, sim_dense_b = dense_retrieve(q_texts_b, N_DENSE, doc_emb)

del emb
torch.cuda.empty_cache()


## 3. Анализ данных
Считаем верхнюю границу, до которой в принципе может дотянуться реранкер

In [ ]:
# реранкинг не достанет то, чего нет в пуле Потолок BM25
pos_ids = valid["item_id"].to_numpy()
k_ret = res_valid.shape[1]
ranks = np.full(len(valid), k_ret, dtype=np.int32)
for i, row in enumerate(res_valid):
    ids_row = corpus_ids[row]
    for j, iid in enumerate(ids_row):
        if iid == pos_ids[i]:
            ranks[i] = j
            break
print("K     recall@K")
for K in [50, 100, 200, 300, 500]:
    print(f"{min(K, k_ret):>4}  {(ranks < min(K, k_ret)).mean():.4f}")

# позитивы vs кандидаты BM25 top-50
q_loc = valid["search_location_id"].to_numpy()
q_cat = valid["search_category"].to_numpy()
pos_rows = np.array([id2row[i] for i in valid["item_id"]])
print(f"\nпозитивы:        локация {np.mean(cf['loc'][pos_rows] == q_loc):.1%}, "
      f"категория {np.mean(cf['cat'][pos_rows] == q_cat):.1%}")
loc_r = [np.mean(cf["loc"][row[:50]] == q_loc[i]) for i, row in enumerate(res_valid)]
cat_r = [np.mean(cf["cat"][row[:50]] == q_cat[i]) for i, row in enumerate(res_valid)]
print(f"кандидаты top-50: локация {np.mean(loc_r):.1%}, категория {np.mean(cat_r):.1%}")

# потолок объединённого пула, по сути реранкинга
hits, sizes = [], []
for i, row in enumerate(res_valid):
    s = set(corpus_ids[row[:N_BM25]])
    s.update(cf["loccat"].get((q_loc[i], q_cat[i]), [])[:M_LOCCAT])
    hits.append(pos_ids[i] in s)
    sizes.append(len(s))
print(f"\nпотолок пула: recall={np.mean(hits):.4f}, средний размер={np.mean(sizes):.0f}")

# потолок объедененного пула из трех
hits, sizes = [], []
for i, row in enumerate(res_valid):
    s = set(corpus_ids[row[:N_BM25]])
    s.update(corpus_ids[res_dense_v[i][:N_DENSE]])
    s.update(cf["loccat"].get((q_loc[i], q_cat[i]), [])[:M_LOCCAT])
    hits.append(pos_ids[i] in s)
    sizes.append(len(s))
print(f"\nпотолок BM25∪E5∪лок×кат: recall={np.mean(hits):.4f}, средний размер={np.mean(sizes):.0f}")


K     recall@K
  50  0.3138
 100  0.4309
 200  0.5695
 300  0.6661
 500  0.7759

позитивы:        локация 84.2%, категория 100.0%
кандидаты top-50: локация 6.0%, категория 99.7%

потолок пула: recall=0.8499, средний размер=666

потолок BM25∪E5∪лок×кат: recall=0.8774, средний размер=829


## 4. Этап 2 — реранкер (CatBoost)

Признаки пары запрос—объявление (по списку NUM_FEATURES) + категориальный microcat

Модель CatBoost YetiRank: групповой ранкинг, группа = запрос, у запроса может быть
несколько подходящих объявлений, бинарные метки

In [ ]:
# запрос - весь набор поисковых признаков, сплит по запросам
groups = valid.groupby(QUERY_KEY_COLS, dropna=False).indices
keys = list(groups.keys())
rng = np.random.RandomState(RANDOM_STATE)
rng.shuffle(keys)
n_train = int(len(keys) * 0.8)
train_keys, val_keys = keys[:n_train], keys[n_train:]
print(f"запросов: {len(keys):,} | train: {len(train_keys):,} | val: {len(val_keys):,}")


запросов: 26,556 | train: 21,244 | val: 5,312


In [ ]:
_rating_re = re.compile(r"Рейтинг пользователя (\d)")

def group_candidates(q_row, bm25_row, bm25_score_row, dense_row, dense_sim_row):
    cand = {}
    for rank in range(min(N_BM25, len(bm25_row))):
        iid = corpus_ids[bm25_row[rank]]
        cand[iid] = [float(bm25_score_row[rank]), float(rank), 1.0, 0.0, 0.0, float(N_DENSE)]
    for iid in cf["loccat"].get((q_row["search_location_id"], q_row["search_category"]), [])[:M_LOCCAT]:
        if iid in cand:
            cand[iid][3] = 1.0
        else:
            cand[iid] = [0.0, float(N_BM25), 0.0, 1.0, 0.0, float(N_DENSE)]
    for rank in range(min(N_DENSE, len(dense_row))):
        iid = corpus_ids[dense_row[rank]]
        sim = float(dense_sim_row[rank])
        if iid in cand:
            cand[iid][4] = sim
            cand[iid][5] = float(rank)
        else:
            cand[iid] = [0.0, float(N_BM25), 0.0, 0.0, sim, float(rank)]
    return cand

def group_features(q_row, cand, q_token_set):
    # матрица признаков
    ids = list(cand.keys())
    rows = np.fromiter((id2row[iid] for iid in ids), dtype=np.int64, count=len(ids))
    bm = np.array([cand[iid] for iid in ids], dtype=np.float32)
    n = len(ids)
    m = _rating_re.search(str(q_row["search_infm_params_text"]))
    q_min_rating = float(m.group(1)) if m else 0.0
    X = np.column_stack([
        bm,
        (cf["loc"][rows] == q_row["search_location_id"]).astype(np.float32),
        (cf["cat"][rows] == q_row["search_category"]).astype(np.float32),
        np.full(n, float(q_row["search_is_delivery_search"]), dtype=np.float32),
        np.full(n, float(len(q_token_set)), dtype=np.float32),
        np.array([len(q_token_set & cf["title_sets"][r]) / max(len(q_token_set), 1) for r in rows],
                 dtype=np.float32),
        np.array([len(cf["title_sets"][r]) for r in rows], dtype=np.float32),
        cf["rating"][rows],
        cf["log_reviews"][rows],
        cf["price_neg"][rows],
        cf["log_price"][rows],
        cf["rating"][rows] - q_min_rating,
    ])
    return ids, X, cf["microcat_codes"][rows]

In [ ]:
def build_dataset(keys, groups_map, res, scores, q_sets, dense_res, dense_sim,
                  only_with_positive, desc):
    item_ids = valid["item_id"].to_numpy()
    cols = {name: [] for name in NUM_FEATURES}
    microcat_parts, labels, gids, meta = [], [], [], []
    skipped = 0
    for gid, key in enumerate(tqdm(keys, desc=desc)):
        positions = groups_map[key]
        q_row = valid.iloc[positions[0]]
        cand = group_candidates(q_row, res[gid], scores[gid], dense_res[gid], dense_sim[gid])
        if len(cand) < 2:
            continue
        positives = set(item_ids[positions])
        if only_with_positive and not (positives & cand.keys()):
            skipped += 1
            continue
        ids, X, mc = group_features(q_row, cand, q_sets[gid])
        y = np.array([1.0 if iid in positives else 0.0 for iid in ids], dtype=np.float32)
        for name, col in zip(NUM_FEATURES, X.T):
            cols[name].append(col)
        microcat_parts.append(mc)
        labels.append(y)
        gids.append(np.full(len(ids), gid, dtype=np.int64))
        meta.append((key, positions, ids))
    df = pd.DataFrame({name: np.concatenate(parts) for name, parts in cols.items()})
    df["microcat"] = np.concatenate(microcat_parts)
    print(f"  {desc}: {len(meta)} запросов, {len(df):,} пар, пропущено: {skipped}")
    return df, np.concatenate(labels), np.concatenate(gids), meta

def make_pool(df, y=None, g=None):
    kwargs = {"label": y, "group_id": g} if y is not None else {}
    return Pool(df[ALL_FEATURES], cat_features=CAT_FEATURES,
                feature_names=ALL_FEATURES, **kwargs)


In [ ]:
def train_model(df, y, g, suffix):
    print(f"\nОбучение CatBoost {suffix}")
    t = time.time()
    model = CatBoostRanker(
        iterations=CB_ITERATIONS,
        learning_rate=CB_LEARNING_RATE,
        depth=6,
        loss_function="YetiRank",
        random_seed=RANDOM_STATE,
        task_type=TASK_TYPE,
        devices="1",
        thread_count=-1,
        nan_mode="Min",
        verbose=100,
        allow_writing_files=False,
    )
    model.fit(make_pool(df, y, g))
    print(f"  обучено за {(time.time() - t) / 60:.1f} мин")
    return model


def evaluate(model, meta, df):
    scores = model.predict(make_pool(df))
    bm = df["bm25_score"].to_numpy()
    heur = df["loc_match"].to_numpy() * (bm.max() + 1.0) + bm
    bm_hits, heur_hits, rr_hits, pool_hits, n_rows = 0, 0, 0, 0, 0
    offset = 0
    for _, positions, ids in meta:
        n = len(ids)
        s = scores[offset:offset + n]
        h = heur[offset:offset + n]
        offset += n
        top50 = {ids[i] for i in np.argsort(-s)[:50]}
        top50_heur = {ids[i] for i in np.argsort(-h)[:50]}
        pool = set(ids)
        for row in positions:
            n_rows += 1
            bm_hits += pos_ids[row] in set(corpus_ids[res_valid[row][:50]])
            heur_hits += pos_ids[row] in top50_heur
            rr_hits += pos_ids[row] in top50
            pool_hits += pos_ids[row] in pool
    print(f"  строк (пар): {n_rows}")
    print(f"  BM25 top-50 (бейзлайн):             {bm_hits / n_rows:.4f}")
    print(f"  правило «локация -> BM25» (без ML): {heur_hits / n_rows:.4f}")
    print(f"  BM25 + CatBoost-реранкинг:          {rr_hits / n_rows:.4f}")
    print(f"  потолок пула:              {pool_hits / n_rows:.4f}")


## 5. Локальная валидация и обучение

Сплит 80/20 по запросам
На отложенных запросах сравниваем четыре варианта: чистый BM25, простое правило, локация BM25 без модели, CatBoost-реранкинг и оракул пула (сколько позитивов
вообще лежит в пуле)


In [17]:
tr_rep = np.array([groups[k][0] for k in train_keys])
val_rep = np.array([groups[k][0] for k in val_keys])

df_tr, y_tr, g_tr, meta_tr = build_dataset(
    train_keys, groups,
    res_valid[tr_rep], scores_valid[tr_rep], [q_token_sets[i] for i in tr_rep],
    res_dense_v[tr_rep], sim_dense_v[tr_rep],
    True, "train-датасет")
df_val, y_val, g_val, meta_val = build_dataset(
    val_keys, groups,
    res_valid[val_rep], scores_valid[val_rep], [q_token_sets[i] for i in val_rep],
    res_dense_v[val_rep], sim_dense_v[val_rep],
    False, "val-датасет")

val_model = train_model(df_tr, y_tr, g_tr, "train-сплит")
evaluate(val_model, meta_val, df_val)



--- Обучение CatBoost (YetiRank, train-сплит, GPU 1) ---
Groupwise loss function. OneHotMaxSize set to 10


Default metric period is 5 because PFound is/are not implemented for GPU
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time
Metric PFound is not implemented on GPU. Will use CPU for metric computation, this could significantly affect learning time


0:	total: 1.75s	remaining: 23m 20s
100:	total: 2m 39s	remaining: 18m 24s
200:	total: 5m 17s	remaining: 15m 45s
300:	total: 7m 54s	remaining: 13m 5s
400:	total: 10m 30s	remaining: 10m 27s
500:	total: 13m 7s	remaining: 7m 50s
600:	total: 15m 43s	remaining: 5m 12s
700:	total: 18m 18s	remaining: 2m 35s
799:	total: 20m 53s	remaining: 0us
  обучено за 21.2 мин
  строк (пар): 6389
  BM25 top-50 (бейзлайн):             0.3337
  правило «локация -> BM25» (без ML): 0.7225
  BM25 + CatBoost-реранкинг:          0.7925
  потолок пула (оракул):              0.8798


## 6. Предсказание для бенчмарка и answer.csv

Пул и признаки считаются ровно так же, как при обучении (тот же порядок фич)


In [ ]:
val_model.save_model("/kaggle/working/catboost_reranker_val.cbm")

q_texts_b = [q + " " + (p or "") for q, p in
             zip(queries["search_query"], queries["search_infm_params_text"])]
q_tokens_b = [tok(t) or ["#"] for t in tqdm(q_texts_b, desc="лемматизация бенчмарка")]
res_b, sc_b = retriever.retrieve(q_tokens_b, k=min(N_BM25, len(corpus_ids)))

cols = {name: [] for name in NUM_FEATURES}
microcat_parts, cand_ids_all, sizes = [], [], []
for i in tqdm(range(len(queries)), desc="признаки"):
    q_row = queries.iloc[i]
    cand = group_candidates(q_row, res_b[i], sc_b[i], res_dense_b[i], sim_dense_b[i])
    ids, X, mc = group_features(q_row, cand, set(q_tokens_b[i]))
    for name, col in zip(NUM_FEATURES, X.T):
        cols[name].append(col)
    microcat_parts.append(mc)
    cand_ids_all.append(ids)
    sizes.append(len(ids))

df_b = pd.DataFrame({name: np.concatenate(parts) for name, parts in cols.items()})
df_b["microcat"] = np.concatenate(microcat_parts)
scores_b = val_model.predict(make_pool(df_b))

predictions = []
offset = 0
for n, ids in zip(sizes, cand_ids_all):
    s = scores_b[offset:offset + n]
    offset += n
    predictions.append([ids[j] for j in np.argsort(-s)[:50]])


лемматизация бенчмарка: 100%|██████████| 2452/2452 [00:00<00:00, 54934.40it/s]


BM25S Retrieve:   0%|          | 0/2452 [00:00<?, ?it/s]

признаки: 100%|██████████| 2452/2452 [00:10<00:00, 228.03it/s]


In [19]:
answer = pd.DataFrame({
    "query_id": queries["query_id"].astype(str),
    "answer": [" ".join(top) for top in predictions],
})

assert len(answer) == len(queries), "число строк не совпадает"
assert answer["query_id"].is_unique, "дубликаты query_id"
assert answer["query_id"].str.len().eq(16).all(), "query_id не 16 символов"
valid_item_ids = set(corpus_ids)
for i, s in enumerate(answer["answer"]):
    ids = s.split()
    assert len(ids) <= 50, f"строка {i}: больше 50 item_id"
    assert len(ids) == len(set(ids)), f"строка {i}: дубликаты item_id"
    assert set(ids).issubset(valid_item_ids), f"строка {i}: item_id не из корпуса"

answer.to_csv(ANSWER_PATH, index=False)
print(f"сохранено: {ANSWER_PATH} ({len(answer)} строк)")
answer.head(3)


сохранено: /kaggle/working/answer.csv (2452 строк)


,query_id,answer
0,70DfDUpwjxB4lzFd,14d535285758af89 355392014208b7bf b92ee8f432ce...
1,JTrdTaZJvSiLPkXj,8f74ccd5549146c6 cbeccbecb1fb8d86 1cc96086d662...
2,LZCZNoVG4AFUkVRJ,d8fce513e4f000a7 edecb3695ab88901 10eb6d6da323...
